# Solving `plasmidproject` with the check widget

*Assemble a plasmid from reads, find its ORFs, and decide which of them are genes.*

This is a week-12 project, so students get the goal and the data but no decomposition and no unit tests. The test file checks only four checkpoints: the assembled plasmid, the candidate ORFs, sensitivity and specificity, and the form of the predictions. Everything between them is the student's own plan. The scorer students use is `genescore.py`, and the reference solution is not shipped in any form.

In [ ]:
import vscodenb
import im_pytest   # registers the %%test magic and check()

## The first two checkpoints

With assembly and candidate finding written, the first two checkpoints pass and the other three functions show as *not defined yet*. The two changes from the earlier projects are the overlap threshold in the assembly (the old code counted anything above 2 bases as an overlap) and keeping only the longest ORF for each stop codon, on both strands.

In [ ]:
%%test plasmidproject
#############################################################
# Reference solution for the plasmid project.
#
# Only the five checkpoint functions have fixed names:
#   assemble_plasmid, find_candidates, sensitivity, specificity,
#   predict_genes
# Everything else is one possible decomposition, built from the
# assembly, ORF, translation and primer projects.
#############################################################

from Bio.Align import PairwiseAligner, substitution_matrices

from genescore import score

codon_map = {'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L', 'TCT': 'S',
             'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'TAT': 'Y', 'TAC': 'Y',
             'TAA': '*', 'TAG': '*', 'TGT': 'C', 'TGC': 'C', 'TGA': '*',
             'TGG': 'W', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
             'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P', 'CAT': 'H',
             'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGT': 'R', 'CGC': 'R',
             'CGA': 'R', 'CGG': 'R', 'ATT': 'I', 'ATC': 'I', 'ATA': 'I',
             'ATG': 'M', 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
             'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K', 'AGT': 'S',
             'AGC': 'S', 'AGA': 'R', 'AGG': 'R', 'GTT': 'V', 'GTC': 'V',
             'GTA': 'V', 'GTG': 'V', 'GCT': 'A', 'GCC': 'A', 'GCA': 'A',
             'GCG': 'A', 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
             'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'}

start_codon = 'ATG'

stop_codons = ['TAG', 'TGA', 'TAA']


#############################################################
# Checkpoint 1: assemble the plasmid (from the assembly project)
#############################################################

def read_data(file_name):
    reads = {}
    with open(file_name) as f:
        for line in f:
            name, seq = line.split()
            reads[name] = seq
    return reads


def get_overlap(left, right):
    for i in range(min(len(left), len(right)), 0, -1):
        if left[-i:] == right[:i]:
            return left[-i:]
    return ''


def get_all_overlaps(reads):
    d = {}
    for name1 in reads:
        d[name1] = {}
        for name2 in reads:
            if name1 != name2:
                d[name1][name2] = len(get_overlap(reads[name1], reads[name2]))
    return d


def find_first_read(overlaps, min_overlap):
    # the first read is the one that no other read overlaps from the left
    for right in overlaps:
        left_overlaps = [overlaps[left][right] for left in overlaps if left != right]
        if max(left_overlaps) < min_overlap:
            return right


def find_key_for_largest_value(d):
    return max(d, key=d.get)


def find_order_of_reads(first_read, overlaps, min_overlap):
    order = [first_read]
    while True:
        right = find_key_for_largest_value(overlaps[order[-1]])
        if overlaps[order[-1]][right] < min_overlap:
            return order
        order.append(right)


def reconstruct_sequence(order, reads, overlaps):
    genome = ''
    for left, right in zip(order[:-1], order[1:]):
        genome += reads[left][:-overlaps[left][right]]
    genome += reads[order[-1]]
    return genome


def assemble_plasmid(file_name, min_overlap=100):
    # The week-10 reads overlapped by tens of bases and the code counted
    # anything above 2 as a real overlap. These reads overlap by 250-400
    # bases, and chance overlaps between 1000-base reads easily reach 3,
    # so the threshold has to go up.
    reads = read_data(file_name)
    overlaps = get_all_overlaps(reads)
    first = find_first_read(overlaps, min_overlap)
    order = find_order_of_reads(first, overlaps, min_overlap)
    return reconstruct_sequence(order, reads, overlaps)


#############################################################
# Checkpoint 2: candidate ORFs on both strands
# (from the ORF, translation and primer projects)
#############################################################

def reverse_complement(seq):
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    result = ''
    for base in reversed(seq):
        result += complement[base]
    return result


def translate_orf(orf):
    protein = ''
    for i in range(0, len(orf) - 2, 3):
        protein += codon_map[orf[i:i+3]]
    return protein


def find_next_stop_codon(seq, start):
    for i in range(start, len(seq) - 2, 3):
        if seq[i:i+3] in stop_codons:
            return i
    return None


def find_longest_orfs(seq):
    # Go through the start codons from left to right. The first start codon
    # that reaches a stop codon gives the longest ORF ending there, so later
    # start codons reaching the same stop codon are nested and skipped.
    longest = {}
    for i in range(len(seq) - 2):
        if seq[i:i+3] == start_codon:
            stop = find_next_stop_codon(seq, i)
            if stop is not None and stop not in longest:
                longest[stop] = i
    return longest


def find_candidates(plasmid):
    candidates = []
    for strand, seq in [('+', plasmid), ('-', reverse_complement(plasmid))]:
        longest = find_longest_orfs(seq)
        for stop in longest:
            start = longest[stop]
            candidates.append({'strand': strand,
                               'start': start,
                               'stop': stop,
                               'protein': translate_orf(seq[start:stop])})
    return candidates


## The finished solution

All five functions written, so every check should pass:

In [ ]:
%%test plasmidproject
#############################################################
# Reference solution for the plasmid project.
#
# Only the five checkpoint functions have fixed names:
#   assemble_plasmid, find_candidates, sensitivity, specificity,
#   predict_genes
# Everything else is one possible decomposition, built from the
# assembly, ORF, translation and primer projects.
#############################################################

from Bio.Align import PairwiseAligner, substitution_matrices

from genescore import score

codon_map = {'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L', 'TCT': 'S',
             'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'TAT': 'Y', 'TAC': 'Y',
             'TAA': '*', 'TAG': '*', 'TGT': 'C', 'TGC': 'C', 'TGA': '*',
             'TGG': 'W', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
             'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P', 'CAT': 'H',
             'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGT': 'R', 'CGC': 'R',
             'CGA': 'R', 'CGG': 'R', 'ATT': 'I', 'ATC': 'I', 'ATA': 'I',
             'ATG': 'M', 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
             'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K', 'AGT': 'S',
             'AGC': 'S', 'AGA': 'R', 'AGG': 'R', 'GTT': 'V', 'GTC': 'V',
             'GTA': 'V', 'GTG': 'V', 'GCT': 'A', 'GCC': 'A', 'GCA': 'A',
             'GCG': 'A', 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
             'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'}

start_codon = 'ATG'

stop_codons = ['TAG', 'TGA', 'TAA']


#############################################################
# Checkpoint 1: assemble the plasmid (from the assembly project)
#############################################################

def read_data(file_name):
    reads = {}
    with open(file_name) as f:
        for line in f:
            name, seq = line.split()
            reads[name] = seq
    return reads


def get_overlap(left, right):
    for i in range(min(len(left), len(right)), 0, -1):
        if left[-i:] == right[:i]:
            return left[-i:]
    return ''


def get_all_overlaps(reads):
    d = {}
    for name1 in reads:
        d[name1] = {}
        for name2 in reads:
            if name1 != name2:
                d[name1][name2] = len(get_overlap(reads[name1], reads[name2]))
    return d


def find_first_read(overlaps, min_overlap):
    # the first read is the one that no other read overlaps from the left
    for right in overlaps:
        left_overlaps = [overlaps[left][right] for left in overlaps if left != right]
        if max(left_overlaps) < min_overlap:
            return right


def find_key_for_largest_value(d):
    return max(d, key=d.get)


def find_order_of_reads(first_read, overlaps, min_overlap):
    order = [first_read]
    while True:
        right = find_key_for_largest_value(overlaps[order[-1]])
        if overlaps[order[-1]][right] < min_overlap:
            return order
        order.append(right)


def reconstruct_sequence(order, reads, overlaps):
    genome = ''
    for left, right in zip(order[:-1], order[1:]):
        genome += reads[left][:-overlaps[left][right]]
    genome += reads[order[-1]]
    return genome


def assemble_plasmid(file_name, min_overlap=100):
    # The week-10 reads overlapped by tens of bases and the code counted
    # anything above 2 as a real overlap. These reads overlap by 250-400
    # bases, and chance overlaps between 1000-base reads easily reach 3,
    # so the threshold has to go up.
    reads = read_data(file_name)
    overlaps = get_all_overlaps(reads)
    first = find_first_read(overlaps, min_overlap)
    order = find_order_of_reads(first, overlaps, min_overlap)
    return reconstruct_sequence(order, reads, overlaps)


#############################################################
# Checkpoint 2: candidate ORFs on both strands
# (from the ORF, translation and primer projects)
#############################################################

def reverse_complement(seq):
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    result = ''
    for base in reversed(seq):
        result += complement[base]
    return result


def translate_orf(orf):
    protein = ''
    for i in range(0, len(orf) - 2, 3):
        protein += codon_map[orf[i:i+3]]
    return protein


def find_next_stop_codon(seq, start):
    for i in range(start, len(seq) - 2, 3):
        if seq[i:i+3] in stop_codons:
            return i
    return None


def find_longest_orfs(seq):
    # Go through the start codons from left to right. The first start codon
    # that reaches a stop codon gives the longest ORF ending there, so later
    # start codons reaching the same stop codon are nested and skipped.
    longest = {}
    for i in range(len(seq) - 2):
        if seq[i:i+3] == start_codon:
            stop = find_next_stop_codon(seq, i)
            if stop is not None and stop not in longest:
                longest[stop] = i
    return longest


def find_candidates(plasmid):
    candidates = []
    for strand, seq in [('+', plasmid), ('-', reverse_complement(plasmid))]:
        longest = find_longest_orfs(seq)
        for stop in longest:
            start = longest[stop]
            candidates.append({'strand': strand,
                               'start': start,
                               'stop': stop,
                               'protein': translate_orf(seq[start:stop])})
    return candidates


#############################################################
# Checkpoint 3: sensitivity and specificity
#############################################################

def sensitivity(counts):
    return counts['TP'] / (counts['TP'] + counts['FN'])


def specificity(counts):
    return counts['TN'] / (counts['TN'] + counts['FP'])


def evaluate(predictions, candidates):
    counts = score(predictions, candidates)
    return sensitivity(counts), specificity(counts)


def length_table(candidates, min_lengths):
    # the baseline every other piece of evidence is compared to
    table = []
    for min_length in min_lengths:
        predictions = [c for c in candidates if len(c['protein']) >= min_length]
        table.append((min_length,) + evaluate(predictions, candidates))
    return table


#############################################################
# Evidence: each function looks at one candidate and returns
# a True/False verdict or a number
#############################################################

def upstream(candidate, plasmid, length=20):
    if candidate['strand'] == '+':
        seq = plasmid
    else:
        seq = reverse_complement(plasmid)
    start = candidate['start']
    return seq[max(0, start - length):start]


def has_shine_dalgarno(candidate, plasmid):
    # a ribosome binding site: a piece of AGGAGG about 5-15 bases upstream
    region = upstream(candidate, plasmid)[:-4]
    for motif in ['GGAG', 'AGGA', 'GAGG']:
        if motif in region:
            return True
    return False


def read_proteins(file_name):
    proteins = {}
    with open(file_name) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                name = line[1:]
                proteins[name] = ''
            else:
                proteins[name] += line
    return proteins


def make_protein_aligner():
    aligner = PairwiseAligner()
    aligner.mode = 'local'
    aligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    return aligner


def best_homology_score(protein, database, aligner):
    best = 0
    for name in database:
        s = aligner.score(protein, database[name])
        if s > best:
            best = s
    return best


#############################################################
# Checkpoint 4: the predictor
#############################################################

def predict_genes(candidates, plasmid):
    # Long ORFs are kept on length alone. Shorter ones need a ribosome
    # binding site upstream as well. This uses no homology, so it runs in a
    # second; see solution_walkthrough.ipynb for what homology adds.
    predictions = []
    for c in candidates:
        length = len(c['protein'])
        if length >= 120:
            predictions.append(c)
        elif length >= 50 and has_shine_dalgarno(c, plasmid):
            predictions.append(c)
    return predictions


## What the evidence is worth

The reference `predict_genes` uses length and a Shine-Dalgarno motif only, so the checkpoint test runs in seconds. The cell below adds homology to `proteins.fasta`, with the threshold set from shuffled proteins rather than from the score, and compares the rules. When this notebook was written it printed sensitivity/specificity of about 0.80/0.979 for length alone, 0.85/0.980 for length plus Shine-Dalgarno, and 0.92/0.977 with homology added. Each run appends to `score_log.txt`.

In [ ]:
from plasmidproject_solution import *
import random

plasmid = assemble_plasmid('plasmid_reads.txt')
candidates = find_candidates(plasmid)
database = read_proteins('proteins.fasta')
aligner = make_protein_aligner()

# About 40 seconds: the 545 candidates of 30 amino acids or more against 367 proteins
homology = {}
for c in candidates:
    if len(c['protein']) >= 30:
        homology[(c['strand'], c['stop'])] = best_homology_score(c['protein'], database, aligner)

# What does chance alone score? Shuffle 60 candidate proteins and search again.
def shuffled(protein):
    letters = list(protein)
    random.shuffle(letters)
    return ''.join(letters)

random.seed(1)
sample = random.sample([c for c in candidates if len(c['protein']) >= 30], 60)
null = sorted(best_homology_score(shuffled(c['protein']), database, aligner) for c in sample)
print('shuffled proteins: highest score', max(null))

def has_homolog(c, threshold=75):
    return homology.get((c['strand'], c['stop']), 0) >= threshold

def combined(c):
    n = len(c['protein'])
    return has_homolog(c) or n >= 120 or (n >= 50 and has_shine_dalgarno(c, plasmid))

for name, rule in [('length >= 100', lambda c: len(c['protein']) >= 100),
                   ('predict_genes (length + SD)', None),
                   ('homology alone', has_homolog),
                   ('homology + length + SD', combined)]:
    if rule is None:
        predictions = predict_genes(candidates, plasmid)
    else:
        predictions = [c for c in candidates if rule(c)]
    sens, spec = evaluate(predictions, candidates)
    print(f'{name:30} sensitivity {sens:.2f}  specificity {spec:.3f}')
